# Utility Regressor: Train and Evaluate

This notebook trains the Ridge and gradient-boosted utility regressors, selects the highest-predicted chain for every held-out circuit, and compares the selections with the oracle chain.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'machine_learning' / 'RL' / 'two_opt_chain_rl_v4').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from machine_learning.supervised_learning.utility_regressor.utility_regressor import (
    DEFAULT_DEPTH_WEIGHT,
    DEFAULT_FEATURE_BATCH,
    DEFAULT_REWARD_TYPE,
    DEFAULT_RUNTIME_WEIGHT,
    DEFAULT_TWO_QUBIT_WEIGHT,
    run_experiment,
)

## 1. Configure the run

Set `RUN_SEARCH=False` for a quick smoke run. Set it to `True` for the full RL-style validation search.

In [ ]:
CSV_PATH = None  # Use dbs/combined_master.csv; set a path to override it.
FEATURE_BATCH = DEFAULT_FEATURE_BATCH
REWARD_TYPE = DEFAULT_REWARD_TYPE
RUNTIME_WEIGHT = DEFAULT_RUNTIME_WEIGHT
TWO_QUBIT_WEIGHT = DEFAULT_TWO_QUBIT_WEIGHT
DEPTH_WEIGHT = DEFAULT_DEPTH_WEIGHT
RANDOM_STATE = 42
RUN_SEARCH = True

## 2. Train both models

The training function reuses the RL v4 circuit-level split. Hyperparameters are selected using mean utility of the chain selected on the validation circuits, then each winning model is refit on the training split.

In [ ]:
results = run_experiment(
    csv_path=CSV_PATH,
    feature_batch=FEATURE_BATCH,
    reward_type=REWARD_TYPE,
    runtime_weight=RUNTIME_WEIGHT,
    two_qubit_weight=TWO_QUBIT_WEIGHT,
    depth_weight=DEPTH_WEIGHT,
    random_state=RANDOM_STATE,
    run_search=RUN_SEARCH,
)

results['bundle']['split_summary']

## 3. Confirm the feature boundary

The model receives initial circuit features and chain metadata only. Reduction, runtime, final metrics, and dynamic progress fields are target/evaluation data, not inputs.

In [ ]:
numeric_features, categorical_features = results['feature_names']
feature_audit = pd.DataFrame({
    'feature_type': ['numeric'] * len(numeric_features) + ['categorical'] * len(categorical_features),
    'feature': numeric_features + categorical_features,
})
feature_audit

## 4. Row-level prediction metrics

These metrics measure utility prediction error for individual circuit-chain rows. The selection metrics below are the main baseline result.

In [ ]:
metric_rows = []
for model_name, values in results['models'].items():
    for split in ('train', 'validation', 'test'):
        metric_rows.append({
            'model': model_name,
            'split': split,
            **values[f'{split}_metrics'],
        })
row_metrics = pd.DataFrame(metric_rows)
row_metrics

## 5. Chain-selection performance

For each circuit, the model scores all available chains and selects one. Regret is the difference between the oracle utility and the utility of the selected chain.

In [ ]:
selection_rows = []
for model_name, values in results['models'].items():
    for split in ('validation', 'test'):
        selection_rows.append({
            'model': model_name,
            'split': split,
            **{
                key: value
                for key, value in values[f'{split}_selection'].items()
                if key != 'selected_chain_counts'
            },
        })
selection_metrics = pd.DataFrame(selection_rows)
selection_metrics

In [ ]:
chain_counts = pd.DataFrame({
    model_name: values['test_selection']['selected_chain_counts']
    for model_name, values in results['models'].items()
}).fillna(0).astype(int)
chain_counts.sort_index()

## 6. Inspect the selected test chains

The table below is useful for checking individual circuit decisions. It includes the predicted and realized utility for the selected chain.

In [ ]:
model_name = 'gradient_boosting'
selected_test = results['models'][model_name]['test_selected'].copy()
selected_test[[
    'original_circuit', 'chain_name', 'predicted_utility', 'utility',
    'total_two_qubit_gate_reduction_pct', 'total_depth_reduction_pct',
    'total_runtime_seconds',
]].head(20)

## 7. View the hyperparameter search results

The search is ranked by validation selected-chain utility, matching the deployment objective.

In [ ]:
for model_name, values in results['models'].items():
    print(model_name)
    display(values['search_results'].head(10))

## Command-line equivalent

From the repository root:

```bash
python machine_learning/supervised_learning/utility_regressor/utility_regressor.py --search
python machine_learning/supervised_learning/utility_regressor/utility_regressor.py --no-search
```

To change the reward setup, add for example `--runtime-weight 0.1 --two-qubit-weight 1.0 --depth-weight 0.1`.